# SpeechSense Phase 2 — WavLM embeddings
This notebook uses a free Colab GPU when available. It reads the existing validated manifest and original WAV files, saves resumable embedding shards to Drive, and trains lightweight classifiers. It does not modify the source audio or inspect the final test split during model selection.

In [ ]:
!pip -q install 'transformers>=4.45,<5' 'scikit-learn>=1.5,<2' librosa soundfile tqdm joblib pandas
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — enable a GPU runtime')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Set `PROJECT_ROOT` to the folder containing this repository in your Drive. Keep the repository structure intact, including `dataset/` and `reports/phase2/dataset_manifest.csv`. No personal path is built into the notebook.

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/CHANGE_ME/Speech-Emotion-Recognition')
assert (PROJECT_ROOT / 'reports/phase2/dataset_manifest.csv').is_file(), 'Set PROJECT_ROOT correctly'
assert (PROJECT_ROOT / 'experiments/phase2/extract_wavlm_embeddings.py').is_file()
%cd {PROJECT_ROOT}
print('Project:', PROJECT_ROOT)

In [ ]:
!python experiments/phase2/extract_wavlm_embeddings.py --device auto --shard-size 64
# Safe to rerun after interruption: completed shards are skipped.

In [ ]:
!python experiments/phase2/train_embedding_classifier.py
# Optional after the fast classifiers finish:
# !python experiments/phase2/train_embedding_classifier.py --include-rbf

In [ ]:
import json
report_path = PROJECT_ROOT / 'reports/phase2/wavlm_classifiers/evaluation.json'
report = json.loads(report_path.read_text())
print('Selected validation model:', report['selected_validation_model'])
for name, result in report['results'].items():
    print(name, 'CV macro F1=', round(result['cross_validation']['macro_f1'], 4), 'validation macro F1=', round(result['validation']['macro_f1'], 4))
print('Final test evaluation:', report['test_evaluation'])